# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and performing exploratory data analysis (EDA) on the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. We'll also inspect dataset-level metadata for a high-level overview.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load croissant dataset and its metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Version: {metadata.version} | Published: {metadata.datePublished}\nIdentifier: {metadata.identifier}")

## 2. Data Overview

Let’s review the available record sets, along with their `@id`s and the fields of each record set. All referencing is performed by `@id`, as per the Croissant specification.

In [ ]:
# List all record sets and their field details, referenced by @id
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"  Name: {rs['name']}")
    print(f"  @id: {rs['@id']}")
    print(f"  Fields:")
    for field in rs['fields']:
        print(f"    - {field['name']} (@id: {field['@id']}) | type: {field.get('dataType','N/A')}")
    print('')

## 3. Data Extraction

Let's load data from a specific record set, referenced by its `@id`. Below, we extract and preview the data for the main clinical record set.

In [ ]:
# Choose the record set to extract ('cr:RecordSet/clinical_data' for example)
# First, let's list all record set IDs for clarity
print("Record Set IDs:")
for rs_id in record_set_ids:
    print(f"  {rs_id}")

# Suppose the main clinical table has the @id 'cr:RecordSet/clinical_data' (adjust this to actual @id if needed)
main_record_set_id = record_set_ids[0]  # Use the first for demo; adjust if structure/IDs reveal otherwise

# Load all record sets into pandas dataframes, by @id
dataframes = {}
for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df

print(f"Columns (fields) for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
print(f"\nPreview of {main_record_set_id}:")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We now apply common data processing steps: filtering, normalization of a numeric field, and analysis by grouping. All fields and columns are referenced by their `@id`s.

In [ ]:
# Select a numeric field using its @id. We'll search for a suitable numeric field to analyze.
clinical_df = dataframes[main_record_set_id]
# Display available fields to pick a numeric one
print("Available columns in clinical data (@id):")
for col in clinical_df.columns:
    print(f"  {col} (dtype: {clinical_df[col].dtype})")

# Let's assume field '@id': 'cr:Field/age_at_second_crc' exists and is numeric.
numeric_field_id = None
for col in clinical_df.columns:
    if "age" in col.lower():
        numeric_field_id = col
        break

if numeric_field_id:
    print(f"\nSelected numeric field for analysis (@id): {numeric_field_id}\n")
    # Filter records: age > 50 (example threshold)
    threshold = 50
    filtered_df = clinical_df[clinical_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field, e.g. '@id': 'cr:Field/sex' if present
    group_field_id = None
    for col in clinical_df.columns:
        if "sex" in col.lower() or "gender" in col.lower():
            group_field_id = col
            break

    if group_field_id:
        print(f"\nGrouping by field (@id): {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_age")
        display(grouped_df)
    else:
        print("No group field (sex/gender) found for grouping.")
else:
    print("No numeric age field found for analysis.")

## 5. Visualization

Let's visualize the age distribution and possibly its relation to categorical variables such as sex or MSI status, using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(clinical_df[numeric_field_id], bins=15, kde=True, color='midnightblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group (sex/gender) if available
    if group_field_id:
        plt.figure(figsize=(7,5))
        sns.boxplot(x=clinical_df[group_field_id], y=clinical_df[numeric_field_id], palette="Set2")
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded dataset metadata and records from the FAIR² Croissant schema using `mlcroissant`.
- Explored primary record sets and fields, referencing all entities by their `@id`.
- Extracted clinical data into pandas DataFrames and performed normalization and group analysis on a numeric age field.
- Visualized key distributions, facilitating further analysis of clinical and molecular features in cancer survivors with second primary colorectal cancer.

Continue exploring the data by referencing other field/column `@id`s of interest! For more info, visit the [mlcroissant documentation](https://github.com/mlcommons/croissant/tree/main/python/mlcroissant).